# combine coupled benchmark results

In [2]:
import os
from PIL import Image, ImageDraw, ImageFont

# ===== LaTeX / Overleaf page dimensions =====
DPI = 600
TEXTWIDTH_INCHES = 6.5
TEXTWIDTH_PX = int(TEXTWIDTH_INCHES * DPI)
CELL_PAD = 20
COLS = 3

# ===== Label styling =====
HEADER_H = 120
HEADER_BG = "white"
HEADER_TEXT = "black"
BORDER_COLOR = "#CFCFCF"


def _load_font(size: int):
    """
    Load a clean sans-serif font (normal weight).
    """
    candidates = [
        "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf",
        "/usr/share/fonts/truetype/liberation2/LiberationSans-Regular.ttf",
        "/usr/share/fonts/truetype/msttcorefonts/Arial.ttf",
    ]
    for fp in candidates:
        if os.path.isfile(fp):
            try:
                return ImageFont.truetype(fp, size=size)
            except Exception:
                pass
    return ImageFont.load_default()


def _make_labeled_tile(img, label, cell_w, cell_h):
    """
    Resize image and add integration-method label above it.
    """
    filt = Image.LANCZOS if img.size[0] >= cell_w else Image.BICUBIC
    resized = img.resize((cell_w, cell_h), filt)

    tile = Image.new("RGB", (cell_w, cell_h + HEADER_H), "white")
    draw = ImageDraw.Draw(tile)

    # Header background
    draw.rectangle([0, 0, cell_w, HEADER_H], fill=HEADER_BG)

    # Border
    draw.rectangle(
        [0, 0, cell_w - 1, cell_h + HEADER_H - 1],
        outline=BORDER_COLOR,
        width=2
    )

    # Paste image
    tile.paste(resized, (0, HEADER_H))

    # Draw label (normal font)
    font = _load_font(42)

    bbox = draw.textbbox((0, 0), label, font=font)
    text_w = bbox[2] - bbox[0]
    text_h = bbox[3] - bbox[1]

    tx = (cell_w - text_w) // 2
    ty = (HEADER_H - text_h) // 2

    draw.text((tx, ty), label, fill=HEADER_TEXT, font=font)

    # Separator line
    draw.line([(0, HEADER_H), (cell_w, HEADER_H)], fill=BORDER_COLOR, width=2)

    return tile


def combine_images_grid_labeled(items, output_path, cols=COLS):
    """
    Combine labeled images into a publication-quality grid.
    """
    images = [Image.open(p).convert("RGB") for p, _ in items]
    labels = [label for _, label in items]

    n = len(images)
    rows = (n + cols - 1) // cols

    cell_w = (TEXTWIDTH_PX - (cols - 1) * CELL_PAD) // cols

    aspects = [img.size[1] / img.size[0] for img in images]
    median_aspect = sorted(aspects)[len(aspects) // 2]
    cell_h = int(cell_w * median_aspect)

    tiles = []
    for img, label in zip(images, labels):
        tiles.append(_make_labeled_tile(img, label, cell_w, cell_h))

    tile_h = cell_h + HEADER_H
    total_w = cols * cell_w + (cols - 1) * CELL_PAD
    total_h = rows * tile_h + (rows - 1) * CELL_PAD

    canvas = Image.new("RGB", (total_w, total_h), "white")

    for i, tile in enumerate(tiles):
        x = (i % cols) * (cell_w + CELL_PAD)
        y = (i // cols) * (tile_h + CELL_PAD)
        canvas.paste(tile, (x, y))

    canvas.save(output_path, dpi=(DPI, DPI), compress_level=1)


# ===== CONFIG =====
base_dir = "/data1/esraa/Thesis-Project/Results/coupled_benchmark/"
tasks = ["task_1", "task_2"]
ti_methods = ["cellrank", "monocle3", "slingshot", "tscan"]

plot_types = [
    "pseudotime_by_group.png",
    "pseudotime_distribution.png",
    "pseudotime_on_embedding.png",
    "stability_hist_edge_jaccard.png",
    "stability_hist_pseudotime_spearman_abs.png",
    "topology_graph_curved.png",
    "topology_matrix_heatmap.png",
    "trajectory_topology_on_embedding.png",
]

IGNORE_DIRS = {
    "merged_results",
    "tables",
    "figures",
    "logs",
    "adata",
    "plots",
    "plots_pub",
}


# ===== MAIN LOOP =====
for task in tasks:
    for ti_method in ti_methods:
        ti_dir = os.path.join(base_dir, task, ti_method)

        if not os.path.isdir(ti_dir):
            print(f"⚠ Missing: {ti_dir}")
            continue

        integration_folders = sorted([
            f for f in os.listdir(ti_dir)
            if os.path.isdir(os.path.join(ti_dir, f)) and f not in IGNORE_DIRS
        ])

        merged_dir = os.path.join(ti_dir, "merged_results")
        os.makedirs(merged_dir, exist_ok=True)

        for plot in plot_types:
            items = []

            for integration in integration_folders:
                plot_path = os.path.join(
                    ti_dir, integration, "figures", plot
                )

                if os.path.isfile(plot_path):
                    # ✅ ONLY integration method name
                    label = integration
                    items.append((plot_path, label))

            if not items:
                print(f"⚠ No plots: {task}/{ti_method}/{plot}")
                continue

            output = os.path.join(merged_dir, f"combined_{plot}")
            combine_images_grid_labeled(items, output, cols=COLS)

            print(f"✓ Saved: {output}")

✓ Saved: /data1/esraa/Thesis-Project/Results/coupled_benchmark/task_1/cellrank/merged_results/combined_pseudotime_by_group.png
✓ Saved: /data1/esraa/Thesis-Project/Results/coupled_benchmark/task_1/cellrank/merged_results/combined_pseudotime_distribution.png
✓ Saved: /data1/esraa/Thesis-Project/Results/coupled_benchmark/task_1/cellrank/merged_results/combined_pseudotime_on_embedding.png
✓ Saved: /data1/esraa/Thesis-Project/Results/coupled_benchmark/task_1/cellrank/merged_results/combined_stability_hist_edge_jaccard.png
✓ Saved: /data1/esraa/Thesis-Project/Results/coupled_benchmark/task_1/cellrank/merged_results/combined_stability_hist_pseudotime_spearman_abs.png
✓ Saved: /data1/esraa/Thesis-Project/Results/coupled_benchmark/task_1/cellrank/merged_results/combined_topology_graph_curved.png
✓ Saved: /data1/esraa/Thesis-Project/Results/coupled_benchmark/task_1/cellrank/merged_results/combined_topology_matrix_heatmap.png
✓ Saved: /data1/esraa/Thesis-Project/Results/coupled_benchmark/task_1/

# combine ti pipeline results gse614

In [1]:
import os
from PIL import Image

# ===== LaTeX / Overleaf page dimensions (publication quality) =====
DPI = 600                        # high-res for print publications
TEXTWIDTH_INCHES = 6.5           # typical \textwidth for A4 with 1-inch margins
TEXTWIDTH_PX = int(TEXTWIDTH_INCHES * DPI)   # 3900 px
CELL_PAD = 20                    # gap between sub-images (px at 600 DPI)
COLS = 3                         # figures per row


def combine_images_grid(image_paths, output_path, cols=COLS):
    """Arrange images in a grid (cols per row), scaled to fit LaTeX \\textwidth.

    - Each sub-image is individually resized with LANCZOS (high-quality anti-aliased).
    - If a source image is smaller than the target cell, it is up-scaled with
      BICUBIC to avoid blocky artefacts from nearest-neighbour interpolation.
    - Output is saved as lossless PNG at ``DPI`` resolution.
    """
    images = [Image.open(p).convert("RGB") for p in image_paths]
    n = len(images)
    rows = (n + cols - 1) // cols

    # Cell dimensions that tile exactly into TEXTWIDTH_PX
    cell_w = (TEXTWIDTH_PX - (cols - 1) * CELL_PAD) // cols

    # Preserve aspect ratio based on median image dimensions (more robust than first-only)
    aspects = [img.size[1] / img.size[0] for img in images]
    median_aspect = sorted(aspects)[len(aspects) // 2]
    cell_h = int(cell_w * median_aspect)

    # Choose resampling filter per image: LANCZOS for downscale, BICUBIC for upscale
    resized = []
    for img in images:
        filt = Image.LANCZOS if img.size[0] >= cell_w else Image.BICUBIC
        resized.append(img.resize((cell_w, cell_h), filt))

    total_w = cols * cell_w + (cols - 1) * CELL_PAD
    total_h = rows * cell_h + (rows - 1) * CELL_PAD

    canvas = Image.new("RGB", (total_w, total_h), "white")
    for i, img in enumerate(resized):
        x = (i % cols) * (cell_w + CELL_PAD)
        y = (i // cols) * (cell_h + CELL_PAD)
        canvas.paste(img, (x, y))

    canvas.save(output_path, dpi=(DPI, DPI), compress_level=1)  # fast, lossless PNG


# ===== CONFIG =====
base_dir = "/data1/esraa/Thesis-Project/Results/Trajectory_Inference/GSE149614"
methods = ["cellrank", "elpigraph", "monocle3", "scorpius", "slingshot", "tscan", "via", "paga"]
plot_types = [
    "pseudotime_by_group.png",
    "pseudotime_distribution.png",
    "pseudotime_on_embedding.png",
    "stability_hist_edge_jaccard.png",
    "stability_hist_pseudotime_spearman_abs.png",
    "topology_graph_curved.png",
    "topology_matrix_heatmap.png",
    "trajectory_topology_on_embedding.png",
]
 
# Discover tasks from the first available method
tasks = sorted(os.listdir(os.path.join(base_dir, methods[0])))

for task in tasks:
    task_dir = os.path.join(base_dir, "merged_results", task)
    os.makedirs(task_dir, exist_ok=True)

    for plot in plot_types:
        paths = [
            os.path.join(base_dir, m, task, "figures", plot)
            for m in methods
            if os.path.isfile(os.path.join(base_dir, m, task, "figures", plot))
        ]

        if not paths:
            print(f"⚠ No figures found for {task} / {plot}, skipping.")
            continue

        output = os.path.join(task_dir, plot)
        combine_images_grid(paths, output, cols=COLS)
        print(f"✓ Saved {output}")

✓ Saved /data1/esraa/Thesis-Project/Results/Trajectory_Inference/GSE149614/merged_results/task1_macrophage_TAM/pseudotime_by_group.png
✓ Saved /data1/esraa/Thesis-Project/Results/Trajectory_Inference/GSE149614/merged_results/task1_macrophage_TAM/pseudotime_distribution.png
✓ Saved /data1/esraa/Thesis-Project/Results/Trajectory_Inference/GSE149614/merged_results/task1_macrophage_TAM/pseudotime_on_embedding.png
✓ Saved /data1/esraa/Thesis-Project/Results/Trajectory_Inference/GSE149614/merged_results/task1_macrophage_TAM/stability_hist_edge_jaccard.png
✓ Saved /data1/esraa/Thesis-Project/Results/Trajectory_Inference/GSE149614/merged_results/task1_macrophage_TAM/stability_hist_pseudotime_spearman_abs.png
✓ Saved /data1/esraa/Thesis-Project/Results/Trajectory_Inference/GSE149614/merged_results/task1_macrophage_TAM/topology_graph_curved.png
✓ Saved /data1/esraa/Thesis-Project/Results/Trajectory_Inference/GSE149614/merged_results/task1_macrophage_TAM/topology_matrix_heatmap.png
✓ Saved /data1

# gse228



In [2]:
import os
from PIL import Image

DPI = 600                        # high-res for print publications
TEXTWIDTH_INCHES = 6.5           # typical \textwidth for A4 with 1-inch margins
TEXTWIDTH_PX = int(TEXTWIDTH_INCHES * DPI)   # 3900 px
CELL_PAD = 20                    # gap between sub-images (px at 600 DPI)
COLS = 3                         # figures per row


def combine_images_grid(image_paths, output_path, cols=COLS):
    """Arrange images in a grid (cols per row), scaled to fit LaTeX \\textwidth.

    - Each sub-image is individually resized with LANCZOS (high-quality anti-aliased).
    - If a source image is smaller than the target cell, it is up-scaled with
      BICUBIC to avoid blocky artefacts from nearest-neighbour interpolation.
    - Output is saved as lossless PNG at ``DPI`` resolution.
    """
    images = [Image.open(p).convert("RGB") for p in image_paths]
    n = len(images)
    rows = (n + cols - 1) // cols

    # Cell dimensions that tile exactly into TEXTWIDTH_PX
    cell_w = (TEXTWIDTH_PX - (cols - 1) * CELL_PAD) // cols

    # Preserve aspect ratio based on median image dimensions (more robust than first-only)
    aspects = [img.size[1] / img.size[0] for img in images]
    median_aspect = sorted(aspects)[len(aspects) // 2]
    cell_h = int(cell_w * median_aspect)

    # Choose resampling filter per image: LANCZOS for downscale, BICUBIC for upscale
    resized = []
    for img in images:
        filt = Image.LANCZOS if img.size[0] >= cell_w else Image.BICUBIC
        resized.append(img.resize((cell_w, cell_h), filt))

    total_w = cols * cell_w + (cols - 1) * CELL_PAD
    total_h = rows * cell_h + (rows - 1) * CELL_PAD

    canvas = Image.new("RGB", (total_w, total_h), "white")
    for i, img in enumerate(resized):
        x = (i % cols) * (cell_w + CELL_PAD)
        y = (i // cols) * (cell_h + CELL_PAD)
        canvas.paste(img, (x, y))

    canvas.save(output_path, dpi=(DPI, DPI), compress_level=1)  # fast, lossless PNG


# ===== CONFIG =====
base_dir = "/data1/esraa/Thesis-Project/Results/Trajectory_Inference/GSE140228"
methods = ["cellrank", "elpigraph", "monocle3", "scorpius", "slingshot", "tscan", "via", "paga"]
plot_types = [
    "pseudotime_by_group.png",
    "pseudotime_distribution.png",
    "pseudotime_on_embedding.png",
    "stability_hist_edge_jaccard.png",
    "stability_hist_pseudotime_spearman_abs.png",
    "topology_graph_curved.png",
    "topology_matrix_heatmap.png",
    "trajectory_topology_on_embedding.png",
]
 
# Discover tasks from the first available method
tasks = sorted(os.listdir(os.path.join(base_dir, methods[0])))

for task in tasks:
    task_dir = os.path.join(base_dir, "merged_results", task)
    os.makedirs(task_dir, exist_ok=True)

    for plot in plot_types:
        paths = [
            os.path.join(base_dir, m, task, "figures", plot)
            for m in methods
            if os.path.isfile(os.path.join(base_dir, m, task, "figures", plot))
        ]

        if not paths:
            print(f"⚠ No figures found for {task} / {plot}, skipping.")
            continue

        output = os.path.join(task_dir, plot)
        combine_images_grid(paths, output, cols=COLS)
        print(f"✓ Saved {output}")

✓ Saved /data1/esraa/Thesis-Project/Results/Trajectory_Inference/GSE140228/merged_results/task3_myeloid_monocyte_to_TAM/pseudotime_by_group.png
✓ Saved /data1/esraa/Thesis-Project/Results/Trajectory_Inference/GSE140228/merged_results/task3_myeloid_monocyte_to_TAM/pseudotime_distribution.png
✓ Saved /data1/esraa/Thesis-Project/Results/Trajectory_Inference/GSE140228/merged_results/task3_myeloid_monocyte_to_TAM/pseudotime_on_embedding.png
✓ Saved /data1/esraa/Thesis-Project/Results/Trajectory_Inference/GSE140228/merged_results/task3_myeloid_monocyte_to_TAM/stability_hist_edge_jaccard.png
✓ Saved /data1/esraa/Thesis-Project/Results/Trajectory_Inference/GSE140228/merged_results/task3_myeloid_monocyte_to_TAM/stability_hist_pseudotime_spearman_abs.png
✓ Saved /data1/esraa/Thesis-Project/Results/Trajectory_Inference/GSE140228/merged_results/task3_myeloid_monocyte_to_TAM/topology_graph_curved.png
✓ Saved /data1/esraa/Thesis-Project/Results/Trajectory_Inference/GSE140228/merged_results/task3_mye